In [1]:
import pandas as pd
import subprocess
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os


In [2]:
def extract_sample_names(vcf_file):
    command = f"bcftools query -l {vcf_file}"
    print(f"Running command: {command}")
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running command: {result.stderr}")
    if result.stdout:
        return result.stdout.splitlines()
    return []


In [3]:
def create_genotype_df(bed_file, vcf_files):
    try:
        print(f"Processing BED file: {bed_file}")
        if not os.path.exists(bed_file):
            print(f"BED file not found: {bed_file}")
            return None

        regions = pd.read_csv(bed_file, sep='\t', header=None, names=['chrom', 'start', 'end'])
        aggregated_data = []
        sample_names = None

        for vcf_file in vcf_files:
            current_sample_names = extract_sample_names(vcf_file)
            if not current_sample_names:
                print(f"No sample names found in VCF file: {vcf_file}")
                continue

            if not os.path.exists(vcf_file):
                print(f"VCF file not found: {vcf_file}")
                continue

            print(f"Processing VCF file: {vcf_file}")
            for index, row in regions.iterrows():
                region = f"{row['chrom']}:{row['start']}-{row['end']}"
                command = f"bcftools view -m2 -M2 -v snps -r {region} {vcf_file} | bcftools query -f '%CHROM %POS %REF %ALT [ %GT ]\\n'"
                print(f"Running command: {command}")
                result = subprocess.run(command, shell=True, capture_output=True, text=True)
                if result.returncode != 0:
                    print(f"Error running command: {result.stderr}")
                if result.stdout:
                    testsnps = result.stdout.splitlines()
                    data = [line.split() for line in testsnps]
                    aggregated_data.extend(data)
                    
                    if sample_names is None:
                        sample_names = current_sample_names

        if not aggregated_data:
            print(f"No SNP data found in the specified regions.")
            return None

        # Calculate the number of genotype columns dynamically
        num_genotype_columns = len(aggregated_data[0]) - 4
        columns = ['CHROM', 'POS', 'REF', 'ALT'] + sample_names[:num_genotype_columns]

        aut_oneid = pd.DataFrame(aggregated_data)
        aut_oneid.columns = columns

        def convert_genotypes(gt):
            if gt == "0/0":
                return 0
            elif gt in ["0/1", "1/0"]:
                return np.random.choice([0, 1])
            elif gt == "1/1":
                return 1
            elif gt == "./.":
                return np.nan
            return np.nan

        for col in aut_oneid.columns[4:]:
            aut_oneid[col] = aut_oneid[col].apply(convert_genotypes)

        aut_oneid.dropna(inplace=True)

        # Drop non-numeric columns
        aut_oneid = aut_oneid.drop(columns=['CHROM', 'POS', 'REF', 'ALT'])

        return aut_oneid
    except Exception as e:
        print(f"Error in create_genotype_df: {str(e)}")
        return None


In [ ]:
# Combine genotype data from all 22 VCF files
bed_file = "/lisc/scratch/admixlab/aigerim/African/pairwisediff/KSP106_overlap.bed"
vcf_files = [f"25KS.48RHG.74comp.HCBP.{i}.recalSNP99.9.recalINDEL99.0.vcf.gz" for i in range(1, 23)]

genotype_df = create_genotype_df(bed_file, vcf_files)
if genotype_df is None:
    print("No valid SNP data found.")
else:
    print("Genotype data extraction complete.")
    print("Shape of genotype_df:", genotype_df.shape)
    display(genotype_df.head())


In [ ]:
display(genotype_df.head())

In [ ]:
genotype_df

In [7]:
genotype_df.to_csv('intro_regions_for_PCA.csv', index=False)

In [ ]:
import pandas as pd
import subprocess
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

# Load the CSV file into a DataFrame
genotype_df = pd.read_csv("intro_regions_for_PCA.csv")

# Display the first few rows of the DataFrame to verify it's loaded correctly
print(genotype_df.head())

In [ ]:
genotype_df

In [33]:
def perform_pca(genotype_df):
    try:
        pca = PCA(n_components=4)
        pca_result = pca.fit_transform(genotype_df.T.values)  # Transpose the DataFrame
        explained_variance = pca.explained_variance_ratio_ * 100
        return pca_result, explained_variance
    except Exception as e:
        print(f"Error in perform_pca: {str(e)}")
        return None, None


In [34]:
def plot_pca(pca_results, explained_variance, labels, output_file=None, font_size=14):
    try:
        # Define groups
        group1_ids = ["HGDP_HGDP01029", "KSP062", "KSP063", "KSP065", "KSP067", "KSP069", "KSP092", "KSP096", "KSP103", "KSP105", "KSP106", "KSP111", "KSP116", "KSP124", "KSP134", "KSP137", "KSP139", "KSP140", "KSP146", "KSP150", "KSP152", "KSP154", "KSP155", "KSP224", "KSP225", "KSP228"]
        group2_ids = ["HG02568", "HG02922", "HG03052", "HGDP_HGDP00927", "HGDP_HGDP01284", "SGDP_LP6005441-DNA_E07", "SGDP_LP6005441-DNA_F07", "SGDP_LP6005442-DNA_A02", "SGDP_LP6005442-DNA_A10", "SGDP_LP6005442-DNA_B02", "SGDP_LP6005442-DNA_B10", "SGDP_LP6005442-DNA_G10", "SGDP_LP6005442-DNA_G11", "SGDP_LP6005442-DNA_H10", "SGDP_SS6004475", "SGDP_SS6004470", "HGDP_DNK02", "NA19017", "PNP010", "PNP011", "PNP012", "PNP013", "PNP014", "PNP030", "PNP031"]

        print(f"Number of PCA results: {len(pca_results)}")
        print(f"Number of labels: {len(labels)}")

        # Filter PCA results and labels to include only those in group1 or group2
        filtered_indices = [i for i, label in enumerate(labels) if label in group1_ids or label in group2_ids]
        filtered_pca_results = pca_results[filtered_indices]
        filtered_labels = [labels[i] for i in filtered_indices]

        if len(filtered_labels) != len(filtered_pca_results):
            print(f"Warning: Length of filtered labels ({len(filtered_labels)}) does not match number of filtered PCA results ({len(filtered_pca_results)}).")
        
        df = pd.DataFrame(filtered_pca_results, columns=['PC1', 'PC2', 'PC3', 'PC4'])
        df['Label'] = filtered_labels

        # Assign colors based on groups and identify special individual
        colors = []
        star_indices = []
        for i, label in enumerate(filtered_labels):
            if label in group1_ids:
                colors.append('red')
            elif label in group2_ids:
                colors.append('blue')
            if label == "KSP106":
                star_indices.append(i)

        fig, axes = plt.subplots(2, 1, figsize=(6, 10))

        # Plot PC1 vs. PC2
        scatter1 = axes[0].scatter(df['PC1'], df['PC2'], c=colors)
        axes[0].set_xlabel(f"PC1 ({explained_variance[0]:.2f}% variance explained)", fontsize=font_size)
        axes[0].set_ylabel(f"PC2 ({explained_variance[1]:.2f}% variance explained)", fontsize=font_size)

        # Set axis limits for PC1 vs. PC2
        axes[0].set_xlim([-30, 65])  # Example limits, adjust as needed
        axes[0].set_ylim([-30, 65])  # Example limits, adjust as needed
        axes[0].set_xticks(range(-30, 65, 10))  # Ticks from -10 to 10 with step of 10
        axes[0].set_yticks(range(-30, 65, 10))  # Ticks from -10 to 10 with step of 10

        # Highlight KSP106 in PC1 vs. PC2
        for index in star_indices:
            axes[0].scatter(df['PC1'][index], df['PC2'][index], c='yellow', edgecolor='black', marker='*', s=200)

        # Plot PC3 vs. PC4
        scatter2 = axes[1].scatter(df['PC3'], df['PC4'], c=colors)
        axes[1].set_xlabel(f"PC3 ({explained_variance[2]:.2f}% variance explained)", fontsize=font_size)
        axes[1].set_ylabel(f"PC4 ({explained_variance[3]:.2f}% variance explained)", fontsize=font_size)

        # Set axis limits for PC3 vs. PC4
        axes[1].set_xlim([-30, 65])  # Example limits, adjust as needed
        axes[1].set_ylim([-30, 65])  # Example limits, adjust as needed
        axes[1].set_xticks(range(-30, 65, 10))  # Ticks from -10 to 10 with step of 10
        axes[1].set_yticks(range(-30, 65, 10))  # Ticks from -10 to 10 with step of 10

        # Highlight KSP106 in PC3 vs. PC4
        for index in star_indices:
            axes[1].scatter(df['PC3'][index], df['PC4'][index], c='yellow', edgecolor='black', marker='*', s=200)

        # Create custom legend
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], marker='o', color='w', label='KhoeSan', markerfacecolor='red', markersize=10),
                           Line2D([0], [0], marker='o', color='w', label='Ref', markerfacecolor='blue', markersize=10),
                           Line2D([0], [0], marker='*', color='yellow', label='KSP106', markerfacecolor='yellow', markersize=10, markeredgecolor='black')]
        axes[0].legend(handles=legend_elements, fontsize=font_size, loc='best')
        axes[1].legend(handles=legend_elements, fontsize=font_size)

        if output_file:
            plt.savefig(output_file)
        else:
            plt.show()
        plt.close()
    except Exception as e:
        print(f"Error in plot_pca: {str(e)}")



In [ ]:
if genotype_df is not None:
    pca_results, explained_variance = perform_pca(genotype_df)
    if pca_results is not None and explained_variance is not None:
        labels = genotype_df.columns.tolist()
        print("PCA complete.")
        print(f"PCA results shape: {pca_results.shape}")
        print(f"Labels shape: {len(labels)}")
    else:
        print("PCA failed.")

In [ ]:
if pca_results is not None and explained_variance is not None:
    plot_pca(pca_results, explained_variance, labels, output_file=None, font_size=12)
else:
    print("No PCA results to plot.")